In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

In [ ]:
# Directories
CODE_DIR = Path(r"D:\StockTwits\Code")
DATA_DIR = Path(r"D:\StockTwits\Data\v1\data\csv")
FIGURES_DIR = Path(r"D:\StockTwits\Figures")
MODEL_DATA_DIR = Path(r"D:\StockTwits\Data")

# File names
INPUT_DATA = MODEL_DATA_DIR / "merged_master.pkl"

# Sample period
SAMPLE_START = '2012-01-01'
SAMPLE_END = '2022-12-31'

# Legacy text-variant switch (the add_text_features builder it referred to was removed on 2026-09-11):
#   None -> the plain all-features predictions; "raw" | "pca" | "supervised" -> the matching
#   predictions_*_input=N_text=<variant>.pkl files written by a model notebook run with that TEXT_VARIANT.
TEXT_VARIANT = None

def find_all_features_file(model_type, text_variant=None):
    """Resolve the 'all features' prediction filename for a model type: the largest
    input-count file on disk (excluding the 2-feature baseline) whose _text=<variant> tag
    matches `text_variant` (files without a tag when text_variant is None), so this doesn't
    need updating whenever the feature set changes."""
    import re
    pattern = re.compile(r"input=(\d+)(?:_text=([A-Za-z]+))?")
    candidates = []
    for p in MODEL_DATA_DIR.glob(f"predictions_{model_type}_input=*.pkl"):
        m = pattern.search(p.stem)
        if m is None or int(m.group(1)) == 2 or m.group(2) != text_variant:
            continue
        candidates.append((int(m.group(1)), p.name))
    if not candidates:
        tag = "" if text_variant is None else f"_text={text_variant}"
        return f"predictions_{model_type}_input=NOT_FOUND{tag}.pkl"
    return max(candidates, key=lambda t: t[0])[1]

# Model registry
# Evaluate every registered model on the same stock-days: rows where any model has no
# prediction are dropped. The text-only model predicts only stock-days with messages, so
# with it registered the common sample is (essentially) the tweeted stock-days.
COMMON_SAMPLE = True

MODELS = {
    'lr_2': 'predictions_linear_regression_input=2.pkl',
    'lr_all': find_all_features_file('linear_regression', TEXT_VARIANT),
    # Text-only OLS on the 384 embedding dimensions (03a/prediction_linear_regression_text_only.ipynb).
    # Distinct model name on purpose: the resolver above never picks it up, so it is registered explicitly.
    'lr_text': 'predictions_linear_regression_textonly_input=384.pkl',
    # Variants from the same notebook: +norm = embed_norm/embed_cos added, _dm = date-de-meaned training,
    # ridge = walk-forward-selected penalty (lambda path in the .json sidecar next to each file).
    'lr_text_n':       'predictions_linear_regression_textonly_input=386.pkl',
    'lr_text_n_dm':    'predictions_linear_regression_textonly_dm_input=386.pkl',
    'ridge_text_n':    'predictions_ridge_textonly_input=386.pkl',
    'ridge_text_n_dm': 'predictions_ridge_textonly_dm_input=386.pkl',
    # Rank-target runs (RANK_TARGET=1): trained on the daily percentile rank of f_cumret1 (minus 0.5).
    # Scale-free metrics (rank correlation, decile spreads) are the ones to read for these.
    'lr_2_rank':            'predictions_linear_regression_rank_input=2.pkl',
    'lr_all_rank':          'predictions_linear_regression_rank_input=53.pkl',
    'lr_text_n_rank':       'predictions_linear_regression_textonly_rank_input=386.pkl',
    'lr_text_n_dm_rank':    'predictions_linear_regression_textonly_dm_rank_input=386.pkl',
    'ridge_text_n_rank':    'predictions_ridge_textonly_rank_input=386.pkl',
    'ridge_text_n_dm_rank': 'predictions_ridge_textonly_dm_rank_input=386.pkl',
}

# Prediction target
TARGET = 'f_cumret1'

# Load and merge data

In [ ]:
# Load base data
df = pd.read_pickle(INPUT_DATA)[['date', 'permno', 'ticker', TARGET, 'log_volume']].copy()

# Merge predictions from each model
for model_name, pred_file in MODELS.items():
    preds = pd.read_pickle(MODEL_DATA_DIR / pred_file)
    df[model_name] = preds.set_index('index')['prediction']

# Keep only rows with at least one prediction (OOS period)
model_cols = list(MODELS.keys())
df = df.dropna(subset=model_cols, how='all').reset_index(drop=True)

# Restrict to sample period
df['date'] = pd.to_datetime(df['date'])
df = df[(df['date'] >= SAMPLE_START) & (df['date'] <= SAMPLE_END)].reset_index(drop=True)
df['year_month'] = df['date'].dt.to_period('M')
df['year'] = df['date'].dt.year

print(f"OOS sample: {len(df):,} rows")
print(f"Period: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Models loaded: {model_cols}")
for col in model_cols:
    print(f"  {col}: {df[col].notna().sum():,} predictions")

if COMMON_SAMPLE:
    n_before = len(df)
    df = df.dropna(subset=model_cols).reset_index(drop=True)
    print(f"COMMON_SAMPLE: dropped {n_before - len(df):,} rows lacking a prediction from some model; remaining {len(df):,}")

# Daily cross-sectional rank correlations

In [ ]:
def daily_rank_corr(group, model_cols, target):
    """Compute Spearman rank correlation between each model's predictions and realized returns for one day."""
    result = {}
    for col in model_cols:
        valid = group[[target, col]].dropna()
        if len(valid) < 10:
            result[col] = np.nan
        else:
            corr, _ = spearmanr(valid[col], valid[target])
            result[col] = corr
    result['N'] = len(group)
    return pd.Series(result)

In [ ]:
daily_corr = df.groupby('date').apply(daily_rank_corr, model_cols=model_cols, target=TARGET)
daily_corr['year'] = daily_corr.index.year
daily_corr['year_month'] = daily_corr.index.to_period('M')

print(f"Daily rank correlations computed for {len(daily_corr):,} days")
print(f"\nSummary statistics:")
print(daily_corr[model_cols].describe())

# Full sample rank correlation

In [ ]:
# Full sample: average of daily cross-sectional rank correlations
full_sample = daily_corr[model_cols].mean()

print("Full Sample Average Daily Rank Correlation (Spearman)")
print("=" * 50)
for col in model_cols:
    print(f"  {col}: {full_sample[col]:.6f}")

# Yearly rank correlation

In [ ]:
# Yearly: average of daily cross-sectional rank correlations within each year
yearly_corr = daily_corr.groupby('year')[model_cols].mean()

print("Yearly Average Daily Rank Correlation (Spearman)")
print("=" * 50)
print(yearly_corr.to_string(float_format='{:.6f}'.format))

# Monthly rank correlation

In [ ]:
# Monthly: average of daily cross-sectional rank correlations within each month
monthly_corr = daily_corr.groupby('year_month')[model_cols].mean()

print("Monthly Average Daily Rank Correlation (Spearman)")
print("=" * 50)
print(monthly_corr.to_string(float_format='{:.6f}'.format))

# Plots

In [ ]:
# Daily rank correlation time series
fig, ax = plt.subplots(figsize=(14, 5))
for col in model_cols:
    ax.plot(daily_corr.index, daily_corr[col], label=col, alpha=0.4, linewidth=0.5)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('Date')
ax.set_ylabel('Spearman Rank Correlation')
ax.set_title('Daily Cross-Sectional Rank Correlation: Predictions vs Realized Returns')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Monthly average rank correlation
fig, ax = plt.subplots(figsize=(14, 5))
x = monthly_corr.index.to_timestamp()
for col in model_cols:
    ax.plot(x, monthly_corr[col], label=col, marker='.', markersize=3)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('Month')
ax.set_ylabel('Spearman Rank Correlation')
ax.set_title('Monthly Average Rank Correlation: Predictions vs Realized Returns')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Yearly average rank correlation
fig, ax = plt.subplots(figsize=(10, 5))
x = yearly_corr.index.astype(int)
for col in model_cols:
    ax.plot(x, yearly_corr[col], marker='o', label=col)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('Spearman Rank Correlation')
ax.set_title('Yearly Average Rank Correlation: Predictions vs Realized Returns')
ax.set_xticks(x)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of daily rank correlations
fig, axes = plt.subplots(1, len(model_cols), figsize=(7 * len(model_cols), 5))
if len(model_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, model_cols):
    vals = daily_corr[col].dropna()
    ax.hist(vals, bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(x=vals.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean = {vals.mean():.4f}')
    ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
    ax.set_xlabel('Spearman Rank Correlation')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Distribution of Daily Rank Correlations: {col}')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()